In [1]:
from mtga_viz import *

In [2]:
# Take all desired tournaments and leagues and merge them into one single dataframe. For tournaments, mirror results to also consider oppo_decks as user_decks for later matrix
df = load_and_merge_data(tournament_directory= '../data/tournament',
                         tournament_files=['2026_04_26.csv', '2026_05_17.csv', '2026_05_28_meta.csv', '2026_06_13.csv', '2026_06_27.csv'],
                         league_directory='../data/league',
                         league_files=['2026_season_2.csv'])

In [3]:
# Clean it
df_clean= clean_raw_csv(df, user_deck_col='user_deck', oppo_deck_col='oppo_deck')
df_clean

,match_created_utc,user_deck,oppo_deck,result_vs_oppo
0,2026-04-26T16:20:23+00:00,br burn,br breach,0-2
1,2026-04-26T17:07:57+00:00,ub tempo,ur tempo,2-1
2,2026-04-26T16:22:43+00:00,mono r ruby storm,ur tempo,0-2
3,2026-04-26T16:38:42+00:00,mono r ruby storm,ub tempo,2-1
4,2026-04-26T16:33:38+00:00,wur tempo,mono w stompy,1-2
...,...,...,...,...
1043,2026-05-15T14:24:16+00:00,death and taxes,ur tempo,1-2
1044,2026-05-15T23:24:19+00:00,ub tempo,mono b stompy,1-2
1045,2026-05-15T16:42:22+00:00,green depths,ubr tempo,2-0
1046,2026-05-15T05:12:06+00:00,br burn,ur tempo,1-2


In [4]:
# Similarly, explore which decks repeat the most! You can also use this to identify if the same deck has different naming and/or to find decks that may belong to the same subfamily.
most_played_decks, common_deck_counting = explore_most_played_decks(df_clean, 'user_deck', 'oppo_deck')
most_played_decks

['wbr energy',
 'ub tempo',
 'ur sneak and show',
 'mono b stompy',
 'ur tempo',
 'rw energy',
 'ub reanimator',
 'mono w stompy',
 'wub tempo',
 'mono r ruby storm',
 'br burn',
 'colorless eldrazi',
 'ub sorin reanimator',
 'ubr tempo',
 'ubg show and tell',
 'ruby storm',
 'mono r stompy',
 'br aggro',
 'br breach',
 'bg lands',
 'ubg raph reanimator',
 'ub doomsday',
 'wu tempo',
 'wub jace valki',
 'ur easel',
 'shredder aggro',
 'death and taxes',
 'ur wizards',
 'ur obosh',
 'ur affinity',
 '4c madness',
 'ub tainted pact',
 'mono g eldrazi',
 'brg deathshadow',
 'turbo reanimator',
 'the epic storm',
 'mono u charbelcher',
 'ub show and tell',
 'mono r burn',
 'raph reanimator',
 'ur phoenix',
 'br belcher',
 'rg goblin foodchain',
 'gu affinity',
 'ur brandon prowess',
 'merfolk',
 'wu painter',
 'green depths',
 'wur tempo',
 'mono b reanimator',
 'wu control',
 'wur oracle control',
 'wg depths',
 'ubg beans',
 'wu stoneforge',
 'wu (kaheera)',
 'brg midrange',
 'ur prowess'

In [5]:

family_decks = {
    "ubx tempo": ["wub tempo", "ubr tempo", "ub tempo"],
    "eldrazi": ['colorless eldrazi', 'colorless', 'mono g eldrazi'],
    "urx tempo": ['ur tempo', 'urg tempo', 'wur tempo'],
    "ruby storm": ['mono r ruby storm', 'ruby storm']
   
}

df_new = relabel_decks(df_clean, user_col='user_deck', oppo_col='oppo_deck', new_names=family_decks)


In [6]:
test = get_table_wr_error(df_new, result_col='result_vs_oppo', study_col='user_deck')
test.head(10)


,user_deck,matches_played,wr,wr_low,wr_high,confidence
44,ubx tempo,117,55.6,46.5,64.2,high
56,wbr energy,112,58.0,48.8,66.8,high
55,urx tempo,73,49.3,38.2,60.5,high
52,ur sneak and show,64,53.1,41.1,64.8,high
18,mono b stompy,62,54.8,42.5,66.6,high
37,ub reanimator,51,47.1,34.1,60.5,high
30,rw energy,51,51.0,37.7,64.1,high
28,ruby storm,44,36.4,23.8,51.1,medium
23,mono w stompy,42,54.8,39.9,68.8,medium
11,eldrazi,30,53.3,36.1,69.8,low


In [7]:
aa, bb, cc, dd = get_matchup_matrix(df_new, 'user_deck', 'oppo_deck', 'result_vs_oppo', top_n=6, min_matches=3)

In [8]:
aa

oppo_deck,mono b stompy,rw energy,ubx tempo,ur sneak and show,urx tempo,wbr energy
user_deck,,,,,,
mono b stompy,NaN,NaN,50.0,83.3,100.0,33.3
rw energy,NaN,NaN,42.9,NaN,50.0,33.3
ubx tempo,50.0,57.1,NaN,66.7,63.0,52.9
ur sneak and show,16.7,NaN,33.3,NaN,100.0,50.0
urx tempo,0.0,50.0,37.0,0.0,NaN,28.6
wbr energy,66.7,66.7,47.1,50.0,71.4,NaN


In [9]:
cc

,user_deck,oppo_deck,matches,wins,losses,wr,wr_masked,confidence
0,mono b stompy,ubx tempo,20,10,10,50.0,50.0,medium
1,mono b stompy,ur sneak and show,12,10,2,83.3,83.3,low
2,mono b stompy,urx tempo,3,3,0,100.0,100.0,very low
3,mono b stompy,wbr energy,18,6,12,33.3,33.3,low
4,rw energy,ubx tempo,14,6,8,42.9,42.9,low
5,rw energy,urx tempo,16,8,8,50.0,50.0,low
6,rw energy,wbr energy,12,4,8,33.3,33.3,low
7,ubx tempo,ur sneak and show,12,8,4,66.7,66.7,low
8,ubx tempo,urx tempo,27,17,10,63.0,63.0,medium
9,ubx tempo,wbr energy,34,18,16,52.9,52.9,high
